# Phenotype generation: doses in therapies statistics

## Dose phenotype processing methodology

| **Category** | **Outlier Capping Method** | **Transformation Method** |
| :--- | :--- | :--- |
| **All Dose Phenotypes** | Outliers are removed in **two consecutive steps** for each phenotype and grouping category (e.g., substance): | **ln Transformation** |
| | **1. Two-sided Percentile-based Outlier Removal:** | |
| | * **Upper Outliers:** Values **above** $4 \times \text{P}_{0.99}$ (the $99^{th}$ percentile) are removed. This handles extreme, non-biological inflation. | |
| | * **Lower Outliers:** Values **below** $\text{P}_{0.01} / 4$ (the $1^{st}$ percentile divided by 4) are removed. This step handles non-biological deflation or spurious low values. | |
| | **2. Sigma-based removal ($8 \times \text{SD}$):** | |
| | The data resulting from Step 1 is then filtered, and any remaining values **above $\mu + 8\sigma$** (mean plus 8 standard deviations) are removed. This ensures the removal of any final extreme outliers before transformation. | |

## Detailed phenotypes definitions

| **Phenotype Name** | **Definition** | **Transformed Phenotype Name** |
| :--- | :--- | :--- |
| **`<drug_name>__max_dose`** | The maximum recorded dose for a specific drug for an individual, capped at the highest dose observed in at least 10 people in the dataset. | **`<drug_name>__max_dose__ln`** |
| **`<drug_name>__optimal_dose`** | The dose associated with the longest continuous period of treatment (therapy duration) for a specific drug. This represents the dose maintained long-term by the patient. | **`<drug_name>__optimal_dose__ln`** |
| **`<drug_name>__mean_dose`** |The average (mean) dose of a specific drug administered to a given individual across all their therapies for that drug. | **`<drug_name>__mean_dose__ln`** |
| **`<drug_name>__median_dose`** | The median dose of a specific drug administered to a given individual across all their therapies for that drug. | **`<drug_name>__median_dose__ln`** |



In [ ]:
import pyspark
import dxpy
import hail as hl
import math
import seaborn as sns
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import sys
import json
import seaborn as sns 
from scipy.stats import boxcox

In [ ]:
sys.path.append('../')
from functions.bnf_dictionaries_utils import prepare_substance_to_bnf_section_code_dict
from functions.phenotype_filtration_and_normalization import cap_outliers_mu_plus_n_sigma, pivot_multiple_phenotypes, phenotype_box_cox_transformation, delete_outliers_above_n_times_percentile, delete_percentile_outliers, delete_outliers_mu_plus_n_sigma
from functions.phenotype_calculation_utils import calculate_therapy_doses

In [ ]:
sc = pyspark.SparkContext()
spark = pyspark.sql.SparkSession(sc)
hl.init(sc=sc, default_reference='GRCh38')

In [ ]:
from datetime import datetime
print(f'Timestamp: {datetime.now()}')
print(f'Instance type: {dxpy.describe(dxpy.JOB_ID)["instanceType"]}')
print(f'Hail version: {hl.version()}')
print(f'Spark version: {spark.version}')

### Configuration and Hail tables loading

In [ ]:
input_database = 'prescriptions_db'
input_prescriptions_tb = 'cleaned_prescriptions_splited_to_therapies_v6.2.0.ht'

#ln_output_database = 'prescriptions_db'
#ln_output_tb = 'ln_doses_v6_2_0.ht'

output_database = 'prescriptions_db'
output_tb = 'doses_values_phenotypes_v6_2_0.ht'

In [ ]:
input_db_id = dxpy.find_one_data_object(name=input_database, classname='database', project=dxpy.PROJECT_CONTEXT_ID)['id']
ht = hl.read_table(f'dnax://{input_db_id}/{input_prescriptions_tb}')
ht = ht.filter(~hl.is_missing(ht.date_struct.year))

In [ ]:
ht_summary = calculate_therapy_doses(ht)

In [ ]:
ht_stats_by_patient_substance = ht_summary.group_by(
    ht_summary.eid, 
    ht_summary.substance
).aggregate(
    max_dose = hl.agg.max(ht_summary.daily_dose),
    #mean_dose = hl.agg.mean(ht_summary.daily_dose),
    median_dose = hl.median(hl.agg.collect(ht_summary.daily_dose)),
    duration_dose_pairs = hl.agg.collect(
        hl.struct(
            duration=ht_summary.duration, 
            dose=ht_summary.daily_dose
        )
    )
)
ht_stats_by_patient_substance = ht_stats_by_patient_substance.persist()

In [ ]:
ht_stats_by_patient_substance = ht_stats_by_patient_substance.annotate(
   sorted_duration_dose_pairs = hl.sorted(
        ht_stats_by_patient_substance.duration_dose_pairs, 
        key=lambda x: -x.duration
   )
)

ht_stats_by_patient_substance = ht_stats_by_patient_substance.persist()

In [ ]:
ht_stats_by_patient_substance = ht_stats_by_patient_substance.annotate(
    sum_weighted_dose = hl.sum(
        ht_stats_by_patient_substance.sorted_duration_dose_pairs.map(
            lambda x: x.duration * x.dose
        )
    ),
    sum_duration = hl.sum(
        ht_stats_by_patient_substance.sorted_duration_dose_pairs.map(
            lambda x: x.duration
        )
    )
)
ht_stats_by_patient_substance = ht_stats_by_patient_substance.annotate(
    mean_dose = hl.if_else(
        ht_stats_by_patient_substance.sum_duration > 0,
        ht_stats_by_patient_substance.sum_weighted_dose / ht_stats_by_patient_substance.sum_duration,
        hl.missing('float64') 
    ),
    optimal_dose = hl.if_else(
        hl.len(ht_stats_by_patient_substance.sorted_duration_dose_pairs) > 0,
        ht_stats_by_patient_substance.sorted_duration_dose_pairs[0].dose,
        hl.missing('float64')
    )
).drop(
    'sorted_duration_dose_pairs', 
    'sum_weighted_dose', 
    'sum_duration'
)
ht_stats_by_patient_substance = ht_stats_by_patient_substance.persist()

In [ ]:
ht_max_dose = ht_stats_by_patient_substance.select('max_dose')
ht_max_dose = delete_percentile_outliers(
    ht=ht_max_dose, 
    grouping_col_name='substance', 
    phenotype_col_name='max_dose',
    percentile_val = 0.01,
    n = 4.0
)
ht_max_dose = ht_max_dose.persist()

ht_mean_dose = ht_stats_by_patient_substance.select('mean_dose')
ht_mean_dose = delete_percentile_outliers(
    ht=ht_mean_dose, 
    grouping_col_name='substance', 
    phenotype_col_name='mean_dose',
    percentile_val = 0.01,
    n = 4.0
)
ht_mean_dose = ht_mean_dose.persist()

ht_median_dose = ht_stats_by_patient_substance.select('median_dose')
ht_median_dose = delete_percentile_outliers(
    ht=ht_median_dose, 
    grouping_col_name='substance', 
    phenotype_col_name='median_dose',
    percentile_val = 0.01,
    n = 4.0
)
ht_median_dose = ht_median_dose.persist()

ht_optimal_dose = ht_stats_by_patient_substance.select('optimal_dose')
ht_optimal_dose = delete_percentile_outliers(
    ht=ht_optimal_dose, 
    grouping_col_name='substance', 
    phenotype_col_name='optimal_dose',
    percentile_val = 0.01,
    n = 4.0
)
ht_optimal_dose = ht_optimal_dose.persist()

In [ ]:
ht_max_dose = delete_outliers_mu_plus_n_sigma(
    ht=ht_max_dose, 
    grouping_col_name='substance', 
    phenotype_col_name='max_dose',
    n = 8.0
)
ht_max_dose = ht_max_dose.persist()

ht_mean_dose = delete_outliers_mu_plus_n_sigma(
    ht=ht_mean_dose, 
    grouping_col_name='substance', 
    phenotype_col_name='mean_dose',
    n = 8.0
)
ht_mean_dose = ht_mean_dose.persist()

ht_median_dose = delete_outliers_mu_plus_n_sigma(
    ht=ht_median_dose, 
    grouping_col_name='substance', 
    phenotype_col_name='median_dose',
    n = 8.0
)
ht_median_dose = ht_median_dose.persist()

ht_optimal_dose = delete_outliers_mu_plus_n_sigma(
    ht=ht_optimal_dose, 
    grouping_col_name='substance', 
    phenotype_col_name='optimal_dose',
    n = 8.0
)
ht_optimal_dose = ht_optimal_dose.persist()

In [ ]:
ht_max_dose = ht_max_dose.annotate(
    ln_max_dose = hl.log(ht_max_dose.max_dose)
)
ht_max_dose = ht_max_dose.persist()

ht_mean_dose = ht_mean_dose.annotate(
    ln_mean_dose = hl.log(ht_mean_dose.mean_dose)
)
ht_mean_dose = ht_mean_dose.persist()

ht_median_dose = ht_median_dose.annotate(
    ln_median_dose = hl.log(ht_median_dose.median_dose)
)
ht_median_dose = ht_median_dose.persist()

ht_optimal_dose = ht_optimal_dose.annotate(
    ln_optimal_dose = hl.log(ht_optimal_dose.optimal_dose)
)
ht_optimal_dose = ht_optimal_dose.persist()

In [ ]:
ln_ht_max_dose = pivot_multiple_phenotypes(
    ht=ht_max_dose, 
    key_col_name='eid', 
    pivot_col_name='substance', 
    phenotype_mappings=[
        ('max_dose', '_max_dose'),
        ('ln_max_dose', '_max_dose__ln')
    ]
)
ln_ht_mean_dose = pivot_multiple_phenotypes(
    ht=ht_mean_dose, 
    key_col_name='eid', 
    pivot_col_name='substance', 
    phenotype_mappings=[
        ('mean_dose', '_mean_dose'),
        ('ln_mean_dose', '_mean_dose__ln')
    ]
)
ln_ht_median_dose = pivot_multiple_phenotypes(
    ht=ht_median_dose, 
    key_col_name='eid', 
    pivot_col_name='substance', 
    phenotype_mappings=[
        ('median_dose', '_median_dose'),
        ('ln_median_dose', '_median_dose__ln')
    ]
)
ln_ht_optimal_dose = pivot_multiple_phenotypes(
    ht=ht_optimal_dose, 
    key_col_name='eid', 
    pivot_col_name='substance', 
    phenotype_mappings=[
        ('optimal_dose', '_optimal_dose'),
        ('ln_optimal_dose', '_optimal_dose__ln')
    ]
)

In [ ]:
final_ht = ln_ht_mean_dose.join(ln_ht_max_dose, how='outer')
final_ht = final_ht.persist()
final_ht = final_ht.join(ln_ht_median_dose, how='outer')
final_ht = final_ht.persist()
final_ht = final_ht.join(ln_ht_optimal_dose, how='outer')
final_ht = final_ht.persist()

In [ ]:
spark.sql(f"CREATE DATABASE IF NOT EXISTS {output_database} LOCATION 'dnax://'")
output_db_id = dxpy.find_one_data_object(name=output_database, classname='database', project=dxpy.PROJECT_CONTEXT_ID)['id']
output_url = f'dnax://{output_db_id}/{output_tb}'

%time final_ht.key_by('eid').write(output_url, overwrite=True)